# Extraction Failure Analysis

Investigates rows where `_extraction_failed = True` in a model output CSV.  
For Azure OpenAI content-filter errors, extracts per-category severity and displays the offending report text.

In [ ]:
import json
import re
import ast
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.size': 11,
    'axes.titlesize': 12,
    'figure.dpi': 150,
})

DATA_DIR      = Path('../data')
FIGURES_DIR = Path('.')

## 1. Load Data

In [ ]:
# ── Select model CSV to inspect ──────────────────────────────────────────────
available = sorted(DATA_DIR.glob('CT-RATE_reports_*.csv'))
print('Available model CSVs:')
for p in available:
    print(' ', p.name)

MODEL_CSV = available[0] if available else None
print(f'\nUsing: {MODEL_CSV}')

In [ ]:
df = pd.read_csv(MODEL_CSV)

n_total   = len(df)
failed    = df[df['_extraction_failed'].fillna(True).astype(bool)].copy()
succeeded = df[~df['_extraction_failed'].fillna(True).astype(bool)].copy()

print(f'Total rows         : {n_total:,}')
print(f'Succeeded          : {len(succeeded):,} ({len(succeeded)/n_total*100:.2f}%)')
print(f'Failed             : {len(failed):,}  ({len(failed)/n_total*100:.2f}%)')
print(f'\nFailed split dist  :')
if 'split' in failed.columns:
    print(failed['split'].value_counts().to_string())

## 2. Parse Error Types

In [ ]:
def classify_error(msg: str) -> str:
    if not isinstance(msg, str):
        return 'unknown'
    if 'content_filter' in msg:
        return 'content_filter'
    if 'api_error' in msg:
        return 'api_error_other'
    if 'json parse failed' in msg or 'JSONDecodeError' in msg:
        return 'json_parse'
    if 'empty' in msg.lower():
        return 'empty_response'
    return 'other'

failed['error_type'] = failed['_parse_error'].apply(classify_error)

print('Error type breakdown:')
print(failed['error_type'].value_counts().to_string())

## 3. Parse Content Filter Details

Extract per-category severity from the Azure OpenAI error message JSON.

In [ ]:
SEVERITY_ORDER = ['safe', 'low', 'medium', 'high']
FILTER_CATEGORIES = ['hate', 'sexual', 'violence', 'self_harm', 'jailbreak']


def extract_content_filter_info(error_msg: str) -> dict:
    """Parse Azure content filter results from the error message string."""
    result = {cat: {'severity': None, 'filtered': None} for cat in FILTER_CATEGORIES}
    result['blocked_categories'] = []

    # The error message embeds a Python-repr dict; extract the JSON-like portion
    # by finding the first '{' after the status code
    match = re.search(r"(\{.*\})", error_msg, re.DOTALL)
    if not match:
        return result

    try:
        # Use ast.literal_eval for Python dict repr inside the error string
        payload = ast.literal_eval(match.group(1))
    except Exception:
        return result

    try:
        filters_list = payload['error']['content_filters']
        for f in filters_list:
            cfr = f.get('content_filter_results', {})
            blocked_cats = []
            for cat in FILTER_CATEGORIES:
                info = cfr.get(cat, {})
                severity = info.get('severity', None)
                filtered = info.get('filtered', False)
                if severity is not None:
                    result[cat] = {'severity': severity, 'filtered': filtered}
                if filtered:
                    blocked_cats.append(cat)
            result['blocked_categories'].extend(blocked_cats)
    except (KeyError, TypeError):
        pass

    return result


cf_rows = []
for idx, row in failed[failed['error_type'] == 'content_filter'].iterrows():
    info = extract_content_filter_info(row['_parse_error'])
    entry = {'index': idx}
    for cat in FILTER_CATEGORIES:
        entry[f'{cat}_severity'] = info[cat]['severity']
        entry[f'{cat}_filtered'] = info[cat]['filtered']
    entry['blocked_categories'] = ', '.join(info['blocked_categories']) or '(none parsed)'
    cf_rows.append(entry)

cf_df = pd.DataFrame(cf_rows).set_index('index')

# Merge back with failed rows for full context
cf_full = failed.loc[cf_df.index].join(cf_df)

print(f'Content-filter failures: {len(cf_full)}')
print('\nBlocked category combinations:')
print(cf_full['blocked_categories'].value_counts().to_string())

In [ ]:
# Severity per category
sev_cols = [f'{c}_severity' for c in FILTER_CATEGORIES]
print('Severity breakdown per category (content-filter failures only):')
for col in sev_cols:
    cat = col.replace('_severity', '')
    counts = cf_full[col].value_counts(dropna=False)
    print(f'  {cat:12s}: {counts.to_dict()}')

### Figure 1 — Content filter severity heatmap

In [ ]:
sev_to_num = {'safe': 0, 'low': 1, 'medium': 2, 'high': 3}

heat_data = cf_full[[f'{c}_severity' for c in FILTER_CATEGORIES]].copy()
heat_data.columns = FILTER_CATEGORIES
heat_data = heat_data.replace(sev_to_num).fillna(0).astype(int)
# Row index: show VolumeName
if 'VolumeName' in cf_full.columns:
    heat_data.index = cf_full['VolumeName'].values

fig, ax = plt.subplots(figsize=(7, max(3, len(heat_data) * 0.55 + 1)))

cmap = plt.get_cmap('RdYlGn_r', 4)
im = ax.imshow(heat_data.values, cmap=cmap, vmin=-0.5, vmax=3.5, aspect='auto')

for i in range(len(heat_data)):
    for j, cat in enumerate(FILTER_CATEGORIES):
        val = heat_data.iloc[i, j]
        label_map = {0: 'safe', 1: 'low', 2: 'med', 3: 'high'}
        filtered = cf_full[f'{cat}_filtered'].iloc[i]
        txt = label_map.get(val, '')
        if filtered:
            txt = f'[{txt}]'  # brackets = filtered/blocked
        ax.text(j, i, txt, ha='center', va='center', fontsize=8,
                color='black' if val < 3 else 'white')

ax.set_xticks(range(len(FILTER_CATEGORIES)))
ax.set_xticklabels(FILTER_CATEGORIES, fontsize=9)
ax.set_yticks(range(len(heat_data)))
ax.set_yticklabels(heat_data.index, fontsize=8)
ax.set_title('Figure 1. Content filter severity per failed report\n(brackets = category blocked)', pad=10)

cbar = plt.colorbar(im, ax=ax, ticks=[0, 1, 2, 3], shrink=0.6)
cbar.ax.set_yticklabels(['safe', 'low', 'medium', 'high'], fontsize=8)

fig.tight_layout()
plt.savefig(FIGURES_DIR / 'fail_fig1_content_filter_heatmap.png', bbox_inches='tight', dpi=200)
plt.show()

### Figure 2 — Count of filtered categories per case

In [ ]:
filter_count = cf_full[[f'{c}_filtered' for c in FILTER_CATEGORIES]].fillna(False).sum(axis=1)
cat_counts   = cf_full[[f'{c}_filtered' for c in FILTER_CATEGORIES]].fillna(False).sum(axis=0)
cat_counts.index = FILTER_CATEGORIES

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Left: how many categories triggered per case
axes[0].bar(filter_count.value_counts().sort_index().index.astype(str),
            filter_count.value_counts().sort_index().values,
            color='#E57373', edgecolor='white')
axes[0].set_xlabel('Number of categories blocked')
axes[0].set_ylabel('Number of failed cases')
axes[0].set_title('Categories blocked per case')
axes[0].yaxis.set_major_locator(plt.MaxNLocator(integer=True))

# Right: which category was most often triggered
axes[1].bar(cat_counts.index, cat_counts.values,
            color='#4878CF', edgecolor='white')
axes[1].set_xlabel('Filter category')
axes[1].set_ylabel('Number of cases blocked')
axes[1].set_title('Frequency of each blocked category')
axes[1].yaxis.set_major_locator(plt.MaxNLocator(integer=True))

fig.suptitle('Figure 2. Content filter trigger patterns', fontsize=12)
fig.tight_layout()
plt.savefig(FIGURES_DIR / 'fail_fig2_filter_patterns.png', bbox_inches='tight', dpi=200)
plt.show()

## 4. Report Text for Each Failed Case

Full Findings + Impressions for every failed row, with filter trigger annotations.

In [ ]:
text_cols = [c for c in ['Findings_EN', 'Impressions_EN'] if c in cf_full.columns]

pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', 200)

SEP = '=' * 80

for i, (idx, row) in enumerate(cf_full.iterrows(), 1):
    vol     = row.get('VolumeName', idx)
    blocked = row.get('blocked_categories', '')
    split   = row.get('split', 'N/A')
    print(SEP)
    print(f'Case {i}/{len(cf_full)} | Volume: {vol} | Split: {split}')
    print(f'Blocked: {blocked}')
    print()
    for col in text_cols:
        print(f'[{col}]')
        print(row.get(col, '(not available)'))
        print()
    print()

## 5. Summary Table

In [ ]:
summary_cols = (['VolumeName', 'split'] + text_cols
                + ['blocked_categories']
                + [f'{c}_severity' for c in FILTER_CATEGORIES])
summary_cols = [c for c in summary_cols if c in cf_full.columns]

summary_df = cf_full[summary_cols].copy()
for col in text_cols:
    summary_df[col] = summary_df[col].str[:120] + '…'  # truncate for display

print('Table 1. Summary of content-filter failures')
display(summary_df.reset_index(drop=True))

## 6. Interpretation & Recommended Action

All failures stem from **Azure OpenAI content filter (HTTP 400)** triggered on the input prompt — not a model logic error.  
The offending content is clinical radiology language describing:
- **`self_harm`** — e.g., reports mentioning suicide attempts, self-inflicted injuries  
- **`violence`** — e.g., reports describing trauma, penetrating injuries, assault-related findings  
- **`hate`** — rare; possibly ethnicity/nationality references in clinical notes  

### Options to recover these rows

| Option | Description |
|---|---|
| **A. Adjust Azure content filter** | In Azure AI Foundry portal, set `self_harm` / `violence` thresholds to "high" (or disable for this deployment). Appropriate for a closed, research-only deployment. |
| **B. Strip triggering sentences before API call** | Pre-process reports to remove or redact sentences that describe injury mechanism; re-run only failed rows. |
| **C. Use a different model/endpoint** | Route failed rows through DeepSeek or another provider without content filtering. |
| **D. Assign default labels (0)** | Accept the 10 failures (~0.04%) as unlabelled; exclude from agreement analysis. |